# Verify method with year 2000

Compare VizHub data for 7 PM2.5-related mortality outcomes in the year 2000 to those calculated using this worklow

In [1]:
import os
import glob
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from utils.mortality_utils import att_frac
from utils.mortality_utils import mortality
import config
from utils.utils import require_dir, create_global_country_map
import pathlib
from utils.utils import land_filter, autosize_figure, standardise_latlon
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.gridspec import GridSpec

In [2]:
# === Path config ===
BMR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR")
POP_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SSP_pop" / "SSP2")
MASK_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country")
PM_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "PM2.5_obs")
TMREL_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "TMREL")
BETA_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "beta_ozone")
BMR_SCRATCH_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "BMR_ozone")

In [3]:
# For file names
GBD_version = "GBD23"

In [4]:
# === Load data ===
pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)
# Calculate average across the "baseline" period
pop_2000 = population.sel(year=slice("1990", "2009")).mean("year")

pm_file = "DIMAQ_PM25_1990-2016.nc"
pm_path = os.path.join(PM_DIR, pm_file)
pm = xr.open_dataset(pm_path)["Mean"]
# Calculate average across the "baseline" period
pm_2000 = pm.sel(year=slice("1990", "2009")).mean("year")
pm_2000 = standardise_latlon(pm_2000)

in_file = "GBD_Country_Masks_0.10.nc"
in_path = os.path.join(MASK_DIR, in_file)
country_mask = xr.open_dataarray(in_path)

In [5]:
# === Health variables ===
# COPD, DIABETES, ISCHEMIC_HEART_DISEASE, LOWER_RESPIRATORY_INFECTIONS,
# LUNG_CANCER, STROKE, DEMENTIA
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER",
               "STROKE", "DEMENTIA"]

## Gridpoint and country level mortality

In [11]:
RR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / GBD_version / "RR_curves")

# TMREL from GBD 2021
TMREL = 4.15  # central estimate [95% Uniform CI 2.4 – 5.9]

for health_VAR in health_vars:
    print(f"Processing mortality outcome {health_VAR}")
    pattern = os.path.join(RR_DIR, f"IHME_GBD_20{GBD_version[-2:]}_AIR_POLLUTION_*_PM_RR_{health_VAR}.nc")
    matches = glob.glob(pattern)
    if len(matches) == 0:
        raise FileNotFoundError(f"No .nc file found for variable: {health_VAR}")
    if len(matches) > 1:
        raise ValueError(f"Multiple .nc files matched for variable {health_VAR}: {matches}")
    RR_values = xr.open_dataset(matches[0])["mean"]

    # --- Scale the RR to the TMREL so that RR below the TMREL=1 ---
    # Updated GBD23 risk curves are log(RR), non updated curves are RR
    if RR_values[0] == 1:
        print("Data starts at 1 so they are 'Relative Risk'")
        # Calculate log(RR)
        logRR = np.log(RR_values)
        if np.any(logRR < 0) is True:
            raise ValueError("Values of logRR < 0, should start at 0")
        # Find the log(RR) at the TMREL
        logRR_tmrel = logRR.sel(exposure=TMREL, method="nearest")
    elif RR_values[0] == 0:
        print("Data starts at 0 so they are 'log(Relative Risk)'")
        # Data is already in logRR format
        logRR = RR_values
        if np.any(logRR < 0) is True:
            raise ValueError("Values of logRR < 0, should start at 0")
        # Find the log(RR) at the TMREL
        logRR_tmrel = logRR.sel(exposure=TMREL, method="nearest")
    else:
        raise ValueError(f"Data has unknown start value: {RR_values[0]}")

    # Shift the function by the log(RR) at the TMREL (logRR_tmrel) so that log(RR)=0 at TMREL
    logRR_shifted = logRR - logRR_tmrel

    # Set log(RR) below TMREL as 0 and exponentiate to get RR
    scaled_RR = np.exp(logRR_shifted.where(logRR_shifted["exposure"] >= TMREL, 0))

    bmr_file = f"{GBD_version}_BMR_Global_Country_Map_{health_VAR}_1990-2009.nc"
    bmr_path = os.path.join(BMR_DIR, bmr_file)
    BMR = xr.open_dataarray(bmr_path)  # central estimate

    # Find RR at each grid point
    RR = scaled_RR.interp(exposure=pm_2000)
    # Calculate the attributable fraction
    AF = 1 - (1/RR)

    M = mortality(AF, BMR, pop_2000)

Processing mortality outcome COPD
Data starts at 1 so they are 'Relative Risk'
Processing mortality outcome DIABETES
Data starts at 1 so they are 'Relative Risk'
Processing mortality outcome ISCHEMIC_HEART_DISEASE
Data starts at 0 so they are 'log(Relative Risk)'
Processing mortality outcome LOWER_RESPIRATORY_INFECTIONS
Data starts at 0 so they are 'log(Relative Risk)'
Processing mortality outcome LUNG_CANCER
Data starts at 1 so they are 'Relative Risk'
Processing mortality outcome STROKE
Data starts at 0 so they are 'log(Relative Risk)'
Processing mortality outcome DEMENTIA
Data starts at 0 so they are 'log(Relative Risk)'


In [13]:
scaled_RR

<xarray.DataArray 'mean' (exposure: 2591)> Size: 21kB
array([1.        , 1.        , 1.        , ..., 1.75838018, 1.75838018,
       1.75838018])
Coordinates:
  * exposure  (exposure) float64 21kB 0.0 0.1 0.2 ... 2.499e+03 2.5e+03